In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import requests
import json
import pandas as pd
from pathlib import Path
import sys
import sqlite3
import re

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    
from extract.config import settings

In [3]:
conn = sqlite3.connect(settings.db_path)
orgs = pd.read_csv("../data/open_secret_top_spenders.csv").loc[:, ['Organization', 'Type']].drop_duplicates()
orgs

,Organization,Type
0,Americans for Constitutional Liberty,501(c)(4)
1,Ohio Works,501(c)(4)
2,Campaign for a Family Friendly Economy,501(c)(4)
3,National Assn of Realtors,501(c)(6)
4,Defending Democracy Together,501(c)(4)
...,...,...
813,Association of Trial Lawyers of America,501(c)(6)
816,Bellco Credit Union,Other
836,Michigan Right to Life,501(c)(4)
845,California Assn Of Realtors,501(c)(6)


In [4]:
pd.read_sql(f"select * from organizations limit 10", conn)

,ein,current_name,normalized_name,first_seen_at,last_seen_at
0,272292010,NORTH CENTRAL ACADEMY,north central academy,2026-07-13 22:45:59,2026-07-20 22:15:08
1,911152733,SHEIKH ABDUL KADIR IDRISS MOSQUE TRUST,sheikh abdul kadir idriss mosque trust,2026-07-13 22:45:59,2026-07-20 22:07:27
2,223519265,FRIENDS OF HOBOKEN CHARTER SCHOOL INC,friends hoboken charter school,2026-07-13 22:45:59,2026-07-20 22:17:13
3,260741074,North Carolina Tax Collectors Association,north carolina tax collectors association,2026-07-13 22:45:59,2026-07-20 22:01:24
4,814522819,CHUA PHAP NGHIEM INC,chua phap nghiem,2026-07-13 22:45:59,2026-07-20 22:00:06
5,473985693,BALDWIN COUNTY HOMEBUILDERS,baldwin county homebuilders,2026-07-13 22:45:59,2026-07-20 22:14:28
6,561576980,ALEXANDER RESCUE SQUAD & EMS INC,alexander rescue squad ems,2026-07-13 22:45:59,2026-07-20 22:10:17
7,223122003,PEDALS FOR PROGRESS,pedals progress,2026-07-13 22:45:59,2026-07-20 21:54:09
8,651156772,EVERGLADES PREPARATORY ACADEMY INC,everglades preparatory academy,2026-07-13 22:45:59,2026-07-20 22:12:30
9,223201959,ATLANTIC COUNTY SPECIAL SERVICES,atlantic county special services,2026-07-13 22:45:59,2026-07-20 22:07:03


In [9]:
type_pattern = re.compile(r'501\(c\)\((\d{1,2})\)')
match_df_cols = ['org', 'org_type', 'ein', 'name', 'exempt_type', 'match_confidence', 'filer_ein', 'filer_name', 'filer_name_norm', 'method', 'verified', 'details']
match_df_rows = []
top_n_matches = 3
for row in orgs.values:
    org = row[0]
    org_type = row[1]
    org_type = type_pattern.search(org_type)
    if org_type:
        org_type = int(org_type.group(1))

    url = f"https://projects.propublica.org/nonprofits/api/v2/search.json?q={org}"
    r = requests.get(url).json()

    if r['total_results'] == 0:
        match_df_row = [
            org, org_type,
            None, None, None, None, None, None, None, # 'ein', 'name', 'exempt_type', 'match_confidence', 'filer_ein', 'filer_name', 'filer_name_norm',
            "auto", 0, "Not found in ProPublica's NonProfit Explorer search method." # 'method', 'verified', 'details'
        ]
        match_df_rows.append(match_df_row)
        continue

    matches = r['organizations']
    matches = [
        [d['ein'], d['name'], d['subseccd'], d['score']]
        for d in matches
    ]

    for i, match in enumerate(matches):
        org_record = pd.read_sql(f"select ein, current_name, normalized_name from organizations where ein = {match[0]}", conn)
        if org_record.empty:
            org_record = [None, None, None] # 'filer_ein', 'filer_name', 'filer_name_norm'
        else:
            org_record = org_record.values.tolist()[0]

        # The first several elements won't change after this point
        match_df_row = [
            org, org_type,
            *match, # 'ein', 'name', 'exempt_type', 'match_confidence'
            *org_record, # 'filer_name', 'filer_name_norm'
        ]

        if i >= top_n_matches:
            match_df_row += [
                "auto", 0, f"Not included in top {top_n_matches} results." # 'method', 'verified', 'details'
            ]
            match_df_rows.append(match_df_row)
            continue

        match_df_row += [
            "manual", None, "" # 'method', 'verified', 'details'
        ]
        match_df_rows.append(match_df_row)
        
match_df = pd.DataFrame(match_df_rows, columns=match_df_cols)
match_df

,org,org_type,ein,name,exempt_type,match_confidence,filer_ein,filer_name,filer_name_norm,method,verified,details
0,Americans for Constitutional Liberty,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,auto,0.0,Not found in ProPublica's NonProfit Explorer s...
1,Ohio Works,4.0,991805086.0,Ohio Works,NaN,97.884690,991805086,Ohio Works,ohio works,manual,NaN,
2,Ohio Works,4.0,820990131.0,Ohio Works,4.0,97.884690,NaN,NaN,NaN,manual,NaN,
3,Ohio Works,4.0,341900652.0,Alliance Of Ohio Work Centers,4.0,62.358486,NaN,NaN,NaN,manual,NaN,
4,Ohio Works,4.0,331202355.0,Ohio School Social Work Association,6.0,62.358486,NaN,NaN,NaN,auto,0.0,Not included in top 3 results.
...,...,...,...,...,...,...,...,...,...,...,...,...
1905,Michigan Right to Life,4.0,351594988.0,Right To Life Michiana Education Fund Inc,3.0,105.467995,351594988,RIGHT TO LIFE MICHIANA EDUCATION,right to life michiana education,auto,0.0,Not included in top 3 results.
1906,Michigan Right to Life,4.0,383144453.0,Right To Life Of Michigan Educational Endowmen...,3.0,104.884580,383144453,RIGHT TO LIFE OF MICHIGAN,right to life michigan,auto,0.0,Not included in top 3 results.
1907,Michigan Right to Life,4.0,382795894.0,Right To Life Of Michigan Educational Foundati...,3.0,104.884580,NaN,NaN,NaN,auto,0.0,Not included in top 3 results.
1908,California Assn Of Realtors,6.0,363535493.0,Institute Of Real Estate Management Of The Nat...,6.0,23.974482,NaN,NaN,NaN,manual,NaN,


In [8]:
match_df.to_csv('../data/potential_dark_money_matches.csv', index=False)